### RFM Analysis

## Load and Clean File

In [17]:
# !pip install pandas openpyxl

import pandas as pd
import numpy as np
import openpyxl

df = pd.read_excel("Online Retail.xlsx")

In [18]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [19]:
df = df.dropna(subset = ['CustomerID'])

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Remove negative Quantities

In [20]:
Pos= df[df['Quantity'] > 0]
Pos.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [21]:
Pos['TotalPrice'] = Pos['Quantity'] * Pos['UnitPrice']
Pos.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [22]:
last_invoice_date = Pos['InvoiceDate'].max()

# 2. Set "now" to one day after that maximum date
snapshot_date = last_invoice_date + pd.Timedelta(days=1)

# 3. Calculate Recency (days since last purchase)
rfm = Pos.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1797.24
3,12349.0,19,1,1757.55
4,12350.0,310,1,334.40


In [23]:
# Rank-based scoring (works even with ties)
rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), 4, labels=[4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1,2,3,4])

# Convert categorical scores to integers
rfm[['R_Score','F_Score','M_Score']] = rfm[['R_Score','F_Score','M_Score']].astype(int)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,12346.0,326,1,77183.60,1,1,4
1,12347.0,2,7,4310.00,4,4,4
2,12348.0,75,4,1797.24,2,3,4
3,12349.0,19,1,1757.55,3,1,4
4,12350.0,310,1,334.40,1,1,2


In [25]:
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

In [26]:
def segment(row):
    if row['RFM_Score'] >= 11:
        return 'Champions'
    elif row['RFM_Score'] >= 9:
        return 'Loyal'
    elif row['RFM_Score'] >= 7:
        return 'Potential'
    elif row['RFM_Score'] >= 5:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment, axis=1)

In [27]:
rfm.head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,12346.0,326,1,77183.60,1,1,4,6,At Risk
1,12347.0,2,7,4310.00,4,4,4,12,Champions
2,12348.0,75,4,1797.24,2,3,4,9,Loyal
3,12349.0,19,1,1757.55,3,1,4,8,Potential
4,12350.0,310,1,334.40,1,1,2,4,Lost
5,12352.0,36,8,2506.04,3,4,4,11,Champions
6,12353.0,204,1,89.00,1,1,1,3,Lost
7,12354.0,232,1,1079.40,1,1,3,5,At Risk
8,12355.0,214,1,459.40,1,1,2,4,Lost
9,12356.0,23,3,2811.43,3,3,4,10,Loyal


In [28]:
rfm['Segment'].value_counts()

Segment
At Risk      999
Champions    868
Potential    856
Lost         809
Loyal        807
Name: count, dtype: int64

In [29]:
rfm.groupby('Segment').agg(Avg_Monetary=('Monetary','mean'), Avg_Recency=('Recency','mean'), Customer_Count=('CustomerID','count'))

,Avg_Monetary,Avg_Recency,Customer_Count
Segment,,,
At Risk,520.480151,113.472472,999
Champions,6773.818353,13.211982,868
Lost,222.342509,229.291718,809
Loyal,1888.271414,38.707559,807
Potential,943.999745,69.948598,856


In [31]:
print("""The Segment 'At Risk Customers' is the largest segment, indicating that they are customers who have not made a purchase recently.
The segment 'Champions' has the highest average monetary value, indicating that they are the most valuable customers in terms of revenue.
The Segments 'Champions' and 'Loyal Customers' are the most valuable segments, as they have high RFM scores and are likely to make repeat purchases. They are the segments that the company should focus on making the VIP program for.
""")

The Segment 'At Risk Customers' is the largest segment, indicating that they are customers who have not made a purchase recently.
The segment 'Champions' has the highest average monetary value, indicating that they are the most valuable customers in terms of revenue.
The Segments 'Champions' and 'Loyal Customers' are the most valuable segments, as they have high RFM scores and are likely to make repeat purchases. They are the segments that the company should focus on making the VIP program for.



In [32]:
email = """ 

To: Marketing Team
Subject: Customer Segmentation Results and Recommendations


Hi Marketing Team,

I have been able to analyze our customer base of about 4,339 customers using the RFM segmentation Recency, Frequency and Monetary value.

Observations:
1. Champions which are about 868 customers are our highest-value segment, with an average of $6,774 and an average last purchase just 13days ago

2. Loyal and Potential customers which are about 807 and 856 respectively are also valuable with average purchases of $1,888 and $944 respectively.

3. "Risk" is our largest segment. They last purchased about 113 days ago on average and spends $520 on average. They are lapsed but not yet lost.

4. Finally, Lost segment customers have very low purchases of about $222 on average and a very long recency of 229 days.

Recommendations:

1. Create a VIP retention program targeting Champions and Loyal customers to reward their loyalty.
2. Launch a win-back campaign for At Risk customers—perhaps with a personalized offer or discount.
3. Avoid spending significant marketing budget on the Lost segment; they are unlikely to respond profitably. Hopefully they see more need for our product.

Let me know if you'd like the full customer list for any of these segments.

Best,
Moses
"""

In [33]:
print(email)

 

To: Marketing Team
Subject: Customer Segmentation Results and Recommendations


Hi Marketing Team,

I have been able to analyze our customer base of about 4,339 customers using the RFM segmentation Recency, Frequency and Monetary value.

Observations:
1. Champions which are about 868 customers are our highest-value segment, with an average of $6,774 and an average last purchase just 13days ago

2. Loyal and Potential customers which are about 807 and 856 respectively are also valuable with average purchases of $1,888 and $944 respectively.

3. "Risk" is our largest segment. They last purchased about 113 days ago on average and spends $520 on average. They are lapsed but not yet lost.

4. Finally, Lost segment customers have very low purchases of about $222 on average and a very long recency of 229 days.

Recommendations:

1. Create a VIP retention program targeting Champions and Loyal customers to reward their loyalty.
2. Launch a win-back campaign for At Risk customers—perhaps with